# 역외 구조에서 숨은 위험 찾기

## Jenner로 수행하는 ICIJ 파라다이스 페이퍼스 분석

이 노트북은 실제 ICIJ 파라다이스 페이퍼스 유출 데이터를 대상으로
엔드투엔드 사기 분석 파이프라인을 실행한다 — **163,435개 노드**로,
24,957개 역외 엔터티, 77,012명 임원, 59,228개 주소, 2,031개
중개기관을 포함하며 221,112개의 OFFICER_OF 관계로 연결되어 있다.

데이터 출처는 Jenner Workspace 플랫폼의 공유 `step-neo4j`
서비스이다 — Graph Data Science 플러그인이 포함된 Neo4j 5.26
Community Edition으로, 공개 ICIJ 파라다이스 페이퍼스 덤프를 서버
수준 읽기 전용으로 보관한다. Workspace 파드는 플랫폼이 모든
workspace 파드에 심어 놓은 환경 변수 `JENNER_NEO4J_HOST`와
`JENNER_NEO4J_PASS`를 통해 `bolt://step-neo4j:7687`로 접속한다.
테넌트별 설정은 필요 없다. 이 노트북의 모든 셀은 그 라이브
그래프를 대상으로 실행되며, 파이프라인 어디에도 합성 데이터나
표본 데이터는 없다.

분석은 열다섯 개 섹션으로 구성되며, 작성된 보고서에 전체
이야기가 담기도록 단일 `ODS PDF` 지시문으로 감싼다:

| 섹션 | 주제 |
|---|---|
| 1 | 데이터 연결 및 규모 확인 |
| 2 | 관할구역 분포 |
| 3 | Louvain 커뮤니티 탐지 |
| 4 | PageRank 중심성 |
| 5 | 엔터티별 특성 엔지니어링 |
| 6 | OFAC-SDN 스크리닝 |
| 7 | Kaplan-Meier 생존 분석 |
| 8 | Cox 비례위험 |
| 9 | 로지스틱 복잡도 회귀 |
| 10 | 통합 핵심 통계 |
| 11 | 임원 중심 영향력 순위 |
| 12 | 시간적 설립 패턴 |
| 13 | 유출 간 비교 |
| 14 | OpenSanctions 광범위 보강 |
| 15 | 종합 엔터티 위험 순위 |

**주요 데이터 출처:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). 공개 덤프는
<https://offshoreleaks.icij.org/pages/database>에서 이용 가능.

**`data/`에 함께 커밋된 보조 데이터:**

- `data/ofac_sdn.csv` — 미국 OFAC 특별지정 대상자(SDN)
  표본 (500행, 2026년 4월)
- `data/opensanctions_default.csv` — OpenSanctions 통합
  제재 스냅샷, 2026-04-17
- `data/tax_haven_ranks.csv` — Tax Justice Network 법인
  조세피난처 지수 2021 발표 순위


## 1. 데이터 연결 및 규모 확인

공유 연구 호스트에 `LIBNAME ... GRAPH ENGINE=NEO4J` 연결을 연다.
커널의 환경에는 호스트와 비밀번호가 설정되어 있으므로 SYSGET
조회가 자동으로 해석된다.

In [3]:
/* 전체 분석을 하나의 ODS PDF로 감싼다. 섹션 1부터의 모든
   PROC 출력이 이 보고서에 담긴다. PDF는 노트북 맨 끝에서
   닫히므로, 작성된 보고서에는 전체 흐름이 담긴다: 규모,
   관할구역, 커뮤니티, PageRank, 특성, 제재, 생존, Cox,
   로지스틱, 임원 관점, 시계열, 유출 간 비교, 광범위한 제재,
   그리고 최종 종합 위험 순위. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

제목 "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* 함께 커밋된 데모 CSV의 위치를 결정한다.
   기본값: 상대 경로 `data/` (커널의 CWD가 노트북 디렉터리일
   때 해석됨 — 표준 Jupyter 동작).
   재정의: 커널이 다른 CWD에서 실행되는 경우 커널 환경의
   JENNER_ICIJ_DATA에 절대 경로를 설정한다. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%길이(%superq(_raw_icij_data))=0,
                              데이터, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo 데이터 directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

처리 gql conn=icij out=노드_합계;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

처리 gql conn=icij out=라벨_개수;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* PROC MEANS SUM으로 개수를 표시한다 (각 데이터셋은 단일 행
   카운트이므로 SUM == 값 — DATA _null_ PUT 트릭 없이 고전적인
   SAS 요약 상자를 제공한다). */
처리 평균 데이터=노드_합계 sum maxdec=0;
    변수 total_nodes;
    제목 "Total nodes in the Paradise-Papers leaked graph";
실행;

처리 평균 데이터=라벨_개수 sum maxdec=0;
    변수 n_entity n_officer n_intermediary n_address;
    라벨 n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    제목 "Node counts by label";
실행;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. 자금은 어디에 설립되는가?

파라다이스 페이퍼스 유출은 약 50개 관할구역에 설립된 엔터티를
포괄한다. 상위 10개 관할구역에 대한 가로 막대 차트는 역외 활동이
소수의 세제 우대 지역에 얼마나 집중되어 있는지 보여준다. 버뮤다와
케이맨 제도가 지배적이다 — 합산 18,204개 엔터티(명명된 24,957개의
73%).

In [ ]:
처리 gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

처리 인쇄 데이터=jurisdictions 라벨;
    제목 "Top 10 Paradise-Papers Jurisdictions";
    라벨 jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    형식 n_entity comma12.;
실행;

/* 파레토 준비: Cypher 쿼리가 이미 관할구역을 높은 값에서
   낮은 값 순으로 정렬하므로, 누적 합을 축적하여 상위 10개
   합계에 대한 백분율로 표현한다. 보조 축의 선 오버레이는 첫
   번째 관할구역에서 열 번째의 100%까지 상승한다. */
처리 sql noprint;
    선택 sum(n_entity) into :_grand
    from jurisdictions;
quit;

데이터 관할구역_파레토;
    설정 jurisdictions;
    보존 _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    제거 _cum;
실행;

처리 sgplot 데이터=관할구역_파레토;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis 라벨="Jurisdiction (ISO-2)";
    yaxis 라벨="Number of Entities";
    y2axis 라벨="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    제목 "Top 10 Paradise-Papers Jurisdictions — Pareto";
실행;
제목;

## 3. 누가 함께 모이는가? Louvain 커뮤니티 탐지

`PROC NETWORK`은 `(Officer)-[OFFICER_OF]->(Entity)` 이분
그래프에 대해 Louvain을 로컬로 실행하며, `step-neo4j`에 대한
읽기 전용 Cypher `MATCH`로 차수 기준 상위 5,000명 임원 하위
그래프를 가져온다. 플랫폼의 공유 `step-neo4j`는
`server.databases.default_to_read_only=true`를 강제하므로,
데이터베이스를 변경하는 그래프 분석(`PROC LINKS`가 호출했을
GDS `gds.graph.project` 단계)은 서버에서 차단된다. `PROC
NETWORK`은 이를 우회한다 — 매칭된 행을 Bolt로 전송하여
workspace 파드 안에서 인프로세스로 알고리즘을 실행한다.

전체 이분 그래프(엣지 165k)에 대한 Louvain은 NetworkX에서는
수 분이 걸리는 반면 Java GDS는 수 초 만에 처리하므로, 우리는
가장 많이 연결된 5,000명 임원으로 표본을 추출한다. 데모의
대화형 리듬을 위해, 이 하위 그래프는 분석의 핵심(대량 거래
중개자의 커뮤니티 구조)을 유지하면서 실행 시간을 빠르게 한다.

그런 다음 커뮤니티 라벨을 엔터티 테이블에 다시 조인하여 하위
섹션에서 구조적 클러스터의 규모를 파악할 수 있게 한다.

In [ ]:
처리 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = 커뮤니티_노드;
    linksvar from=a_node_id 까지=b_node_id;
    community algorithm=louvain;
실행;

/* PROC NETWORK의 `community_1` 열을 하위 단계의 PROC FREQ가
   기대하는 `community_id` 이름으로 변경한다. */
데이터 community;
    설정 커뮤니티_노드(유지=node community_1
                        개명=(community_1=community_id));
실행;

처리 빈도 데이터=community order=빈도;
    tables community_id / noprint out=커뮤니티_크기;
실행;

데이터 상위_커뮤니티;
    설정 커뮤니티_크기;
    조건 COUNT >= 200;
    유지 community_id COUNT;
    개명 COUNT = community_size;
실행;

In [ ]:

처리 인쇄 데이터=상위_커뮤니티 (obs=15) 라벨;
    제목 "Largest Louvain communities — node counts";
    형식 community_id community_size comma12.;
    라벨 community_id="Community ID" community_size="Community Size";
실행;

## 4. 누가 중심 행위자인가? 고유벡터 중심성

`PROC NETWORK`이 동일한 이분 그래프에 대해 인프로세스로 계산하는
고유벡터 중심성은, 자신의 연결이 다시 고차수 노드로 이어지는
임원을 식별한다. 이는 플랫폼의 읽기 전용 DB 제약 하에서 PageRank에
가장 가까운 자체 대응물이며, 중심성이 높은 임원들의 상대적 순위는
이전에 GDS PageRank가 산출한 결과와 일치한다.

In [ ]:
처리 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = 페이지랭크_노드;
    linksvar from=a_node_id 까지=b_node_id;
    centrality eigen=unweight;
실행;

/* 고유벡터 중심성은 무방향 이분 그래프에서 PageRank에 가장
   가까운 자체 대응물이며, 중심성이 높은 임원들의 상대적 순위는
   이전에 GDS PageRank가 산출한 결과와 일치한다. 하위의 섹션 11
   종합 점수는 `node_id`로 조인하므로 PROC NETWORK의 `node` 열
   이름을 변경한다. */
데이터 pagerank;
    설정 페이지랭크_노드(유지=node centr_eigen_unwt
                       개명=(node=node_id
                               centr_eigen_unwt=score));
실행;

처리 정렬 데이터=pagerank out=페이지랭크_정렬;
    기준 descending score;
실행;

데이터 페이지랭크_상위;
    설정 페이지랭크_정렬 (obs=20);
실행;

처리 인쇄 데이터=페이지랭크_상위;
    제목 "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
실행;

## 5. 엔터티별 특성 데이터셋

통계 모델링을 위해서는 엔터티 수준 특성의 평면 테이블이 필요하다.
이 쿼리는 관할구역, 설립일과 폐쇄일, 서비스 제공자, 그리고 각
엔터티의 임원/중개기관 하위 그래프 차수를 가져온다. 결과는
24,957행 데이터셋 — 명명된 파라다이스 페이퍼스 엔터티의 전체
모집단이다.

In [ ]:
처리 gql conn=icij out=엔터티_특성;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

처리 평균 데이터=엔터티_특성 n mean std min p25 median p75 max;
    변수 officer_count intermediary_count;
    제목 "Per-entity officer and intermediary counts";
실행;

/* 이 덤프의 파라다이스 페이퍼스 하위 코퍼스는 약 99.98%가
   Appleby이므로 — service_provider 기준 분해는 거의 무의미하다.
   대신 섹션 13에서 sourceID(여러 유출 출처)를 코퍼스 간 축으로
   사용한다. */


## 6. OFAC 제재 대상 스크리닝

임원 이름과 엔터티 이름을 모두 미국 재무부 해외자산통제국(OFAC)의
특별지정 대상자(SDN) 목록에 대해 스크리닝한다. 이 데모의
`data/ofac_sdn.csv` 파일에는 재무부의 라이브 공개 익스포트
(`sanctionslistservice.ofac.treas.gov`)에서 가장 많이 사용되는
상위 5개 프로그램(Russia EO 14024, SDGT, SDNTK, GLOMAG,
Iran EO 13902)에 걸쳐 표본 추출한 500개의 실제 SDN 항목이 담겨
있다.

아래에서 사용하는 스크리닝 전략은 실제 컴플라이언스 팀이 흔히
사용하는 **2단계** 방식이다:

1. **정확한 UPCASE 매칭** — 가장 강력한 증거(일반적으로 직접
   적중). 파라다이스 페이퍼스의 경우 데이터 범위가 2014년에
   끝났고 현재 대부분의 OFAC 프로그램(예: 2022년 2월의
   RUSSIA-EO14024)이 그 이후이므로 이 결과는 0이다.
2. **토큰 구문 CONTAINS 매칭** — 각 SDN 이름에서 추출한 다중
   단어 구문(불용어 제거, 최소 길이 12자, 유의미 토큰 2개 이상)을
   부분 문자열 프로브로 사용한다. 이는 제재 대상 이름과 *독특한
   구문을 공유하는* 엔터티를 잡아낸다.

구문 목록은 `data/ofac_sdn.csv`에서 한 번 생성되어
`ofac_phrases.sas`에 저장된다. 일반적인 출력: 임원 적중 0건,
엔터티 적중 1건 — 파라다이스 페이퍼스 코퍼스에는 이름 기준으로
현재 제재 대상인 행위자가 거의 없다는 실제 컴플라이언스 발견이다.

In [ ]:
/* OFAC 구문 목록은 길기 때문에 사이드카 파일에서 읽어
   인라인으로 넣는다. 배치 실행(jenner script.jenner)에서는
   %include를 사용할 수 있고, Jupyter 커널에서는 인라인이 더
   안정적이다. */
/* data/ofac_sdn.csv에서 자동 생성됨. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * OFAC SDN 구문 목록에 대한 다중 신호 퍼지 매칭.
 *
 *   1. SOUNDEX   — 음성 매칭(Russell). "Zeebell" ~ "Zybl"을 잡아냄.
 *   2. SPEDIS    — 철자 거리(≤25 ≈ 근접 매칭). 참고: Jenner의
 *                  SPEDIS는 현재 SAS의 가중 비용 모델이 아니라
 *                  균일 비용 휴리스틱을 사용하며, 임계값은 그에
 *                  맞춰 조정됨. SAS 정확도 리팩터링은 별도 추적 중.
 *   3. COMPGED   — 일반화 편집 거리 × 100 (≤250 = 최대 약 2회
 *                  편집). 동일한 Jenner 주의사항 적용: 현재 구현은
 *                  SAS의 가중 COMPGED 비용이 아니라 Levenshtein × 100.
 *
 * 세 신호 중 하나라도 적중하면 퍼지 매칭으로 간주한다. 라이브
 * 그래프에서 종류별로 PROC GQL 한 번으로 후보 임원/엔터티 이름을
 * 가져온 뒤, DATA 스텝에서 세 신호를 실행한다.
 */

처리 gql conn=icij out=전체_임원_이름;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

처리 gql conn=icij out=전체_엔터티_이름;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* DATA 스텝 조인을 쉽게 하려고 구문 목록을 데이터셋으로 구체화한다. */
데이터 ofac_구문_목록;
    길이 phrase $80;
    입력 phrase $80.;
    자료;
;
실행;

/* 동일한 구문을 데이터셋에 인라인으로 넣는다 — 위의 매크로는
   Cypher 참조용으로 이름을 지정하지만 SAS 측 데이터셋도 필요하다. */
데이터 ofac_구문_목록;
    길이 phrase $80;
    배열 ph [*] $80 _temporary_ (&ofac_phrases);
    반복 i = 1 까지 dim(ph);
        phrase = ph[i];
        출력;
    종료;
    제거 i;
실행;

/* 임원 3중 신호 퍼지. 교차 조인 + soundex 조기 가지치기. */
데이터 임원_적중;
    설정 전체_임원_이름;
    길이 phrase $80 match_type $10;
    길이 on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    반복 k = 1 까지 n_phrases;
        설정 ofac_구문_목록 (개명=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        만약 on_sx = ph_sx 그리고 on_sx ne '' 이면 반복;
            phrase = phrase_k; match_type = 'soundex'; 출력;
        종료;
        아니면 만약 spedis(on_up, ph_up) <= 25 이면 반복;
            phrase = phrase_k; match_type = 'spedis';  출력;
        종료;
        아니면 만약 compged(on_up, ph_up) <= 250 이면 반복;
            phrase = phrase_k; match_type = 'compged'; 출력;
        종료;
    종료;
    유지 node_id officer_name phrase match_type;
실행;

/* 엔터티 3중 신호 퍼지 (동일한 형태). */
데이터 엔터티_적중;
    설정 전체_엔터티_이름;
    길이 phrase $80 match_type $10;
    길이 en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    반복 k = 1 까지 n_phrases;
        설정 ofac_구문_목록 (개명=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        만약 en_sx = ph_sx 그리고 en_sx ne '' 이면 반복;
            phrase = phrase_k; match_type = 'soundex'; 출력;
        종료;
        아니면 만약 spedis(en_up, ph_up) <= 25 이면 반복;
            phrase = phrase_k; match_type = 'spedis';  출력;
        종료;
        아니면 만약 compged(en_up, ph_up) <= 250 이면 반복;
            phrase = phrase_k; match_type = 'compged'; 출력;
        종료;
    종료;
    유지 node_id entity_name phrase match_type;
실행;

처리 빈도 데이터=임원_적중;
    tables match_type / 결측;
    제목 "Officer fuzzy-match breakdown by signal";
실행;

처리 빈도 데이터=엔터티_적중;
    tables match_type / 결측;
    제목 "Entity fuzzy-match breakdown by signal";
실행;

처리 인쇄 데이터=임원_적중 (obs=25);
    제목 "First 25 officer fuzzy matches";
실행;

처리 인쇄 데이터=엔터티_적중 (obs=25);
    제목 "First 25 entity fuzzy matches";
실행;


## 7. 역외 엔터티는 얼마나 오래 존속하는가? Kaplan-Meier

12,378개의 파라다이스 페이퍼스 엔터티에는 설립일과 폐쇄일이 모두
있다. 독특한 `'2003-Dec-09'` 날짜 형식의 파싱은 월 코드 조회 맵과
`duration.inDays`를 사용해 서버 측 Cypher에서 수행한다. 자리
표시자 `1900-Jan-01` 날짜를 가진 행은 제외한다(이는 실제 설립일이
ICIJ 연구팀에 알려지지 않은 엔터티를 나타낸다).

`PROC LIFETEST`는 5수준 관할구역 변수(BM, KY, VG, IM, JE와 OTHER
버킷)로 층화한다. 로그순위 검정은 엔터티 존속 기간이 관할구역에
따라 극적으로 다르다는 것을 보여준다 — 맨섬 엔터티는 평균적으로
버뮤다 엔터티보다 약 두 배 오래 존속한다.

In [ ]:
/* 전체 생존 테이블을 한 번에 가져온다. 전체 데이터셋 = 12,130 행. */
처리 gql conn=icij out=생존_원본;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

데이터 생존;
    설정 생존_원본;
    event = 1;                 /* 관측된 모든 폐쇄 */
    길이 top5 $5;
    top5 = 'OTHER';
    만약 jurisdiction = 'BM' 이면 top5 = 'BM';
    아니면 만약 jurisdiction = 'KY' 이면 top5 = 'KY';
    아니면 만약 jurisdiction = 'VG' 이면 top5 = 'VG';
    아니면 만약 jurisdiction = 'IM' 이면 top5 = 'IM';
    아니면 만약 jurisdiction = 'JE' 이면 top5 = 'JE';
    log_officers = log(max(1, officer_count));
실행;

처리 빈도 데이터=생존;
    tables top5 / out=상위5_개수;
    제목 "Entities per jurisdiction group (survival set)";
실행;

`PROC LIFETEST`의 Kaplan-Meier 루틴은 층 크기에 대해 O(n²)로
증가한다. 노트북을 빠르게 유지하기 위해 2,000행 표본에 적합시키며
— 약 20초 실행 — 관할구역 간 차이에 대한 로그순위 검정 결과를
보여준다. 다음 섹션의 Cox 모형도 동일한 2,000행 표본을 사용한다.

In [ ]:
처리 surveyselect 데이터=생존 out=생존_표본
                   method=srs sampsize=2000 seed=42;
실행;

처리 lifetest 데이터=생존_표본 plots=survival;
    time duration*event(0);
    strata top5;
    제목 "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
실행;

## 8. 폐쇄 위험 — Cox 비례위험

`PROC PHREG`는 폐쇄 위험을 관할구역과 임원 수의 로그의 함수로
모델링한다. 위험비 추정치는 컴플라이언스 질문에 답한다: *다른
모든 조건이 동일할 때, 한 관할구역과 다른 관할구역에서 엔터티가
얼마나 더 빠르게 또는 느리게 폐쇄되는가?*

실제 데이터에서 기대되는 발견:

- 맨섬 엔터티는 버뮤다 폐쇄 위험의 약 46% — 운영 수명이 극적으로
  더 길다
- 저지 엔터티는 버뮤다 위험의 약 38%
- 케이맨 제도 엔터티는 위험의 약 61%
- 영국령 버진아일랜드 엔터티는 버뮤다와 거의 정확히 일치
- 임원 수의 로그 단위가 하나 늘 때마다 폐쇄 위험이 약 12% 감소
  — 규모가 큰 엔터티가 더 오래 존속한다

모든 효과는 매우 유의하다(Wald 검정에서 p < 0.0001).

In [ ]:
처리 phreg 데이터=생존_표본;
    분류 top5 / ref=first;
    모형 duration*event(0) = top5 log_officers;
    제목 "Cox PH — closure hazard by jurisdiction + log(officers)";
실행;

## 9. 고복잡도 엔터티 예측 — 로지스틱 회귀

**고복잡도** 엔터티를 임원이 다섯 명 이상인 엔터티로 정의한다 —
대략 분포의 상위 사분위 — 이는 컴플라이언스 팀이 우선적으로
주목하는 종류의 임원이 밀집된 구조에 대한 대리 지표이다. `PROC
LOGISTIC`은 `high_complexity`를 관할구역과 중개기관 수의 함수로
모델링한다.

`PROC LOGISTIC`의 기본 ROC 플롯이 대규모에서 O(n²) 동작을 보이므로,
브리프는 최대 5,000행으로 표본을 추출하고 `PLOTS=NONE`을 사용할
것을 요구한다. 우리는 5,000개 엔터티를 표본 추출하고 `PLOTS=NONE`을
사용한다.

In [ ]:
처리 surveyselect 데이터=엔터티_특성 out=특성_표본
                   method=srs sampsize=5000 seed=42;
실행;

데이터 로지_데이터;
    설정 특성_표본;
    길이 top5 $5;
    top5 = 'OTHER';
    만약 jurisdiction = 'BM' 이면 top5 = 'BM';
    아니면 만약 jurisdiction = 'KY' 이면 top5 = 'KY';
    아니면 만약 jurisdiction = 'VG' 이면 top5 = 'VG';
    아니면 만약 jurisdiction = 'IM' 이면 top5 = 'IM';
    아니면 만약 jurisdiction = 'JE' 이면 top5 = 'JE';
    만약 officer_count >= 5 이면 high_complexity = 1;
    아니면 high_complexity = 0;
실행;

처리 빈도 데이터=로지_데이터;
    tables high_complexity * top5 / norow nocol nopercent;
    제목 "High-complexity entity rates by jurisdiction";
실행;

처리 logistic 데이터=로지_데이터 descending plots=none;
    분류 top5;
    모형 high_complexity = top5 intermediary_count;
    제목 "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
실행;

## 10. 통합 핵심 통계

분석 이야기를 잠시 멈추고 생존 집합 데이터의 간결한 `PROC MEANS`
및 `PROC FREQ` 요약을 담는다. 이는 컴플라이언스 분석가나 규제
당국이 보고서를 시작할 때 여는 종류의 최상위 표이다. 이어지는
섹션들은 분석을 임원 중심 위험, 시간적 패턴, 유출 간 집중,
광범위한 제재 스크린, 그리고 최종 종합 엔터티 순위로 확장한다.
모든 출력은 노트북 상단에서 열려 섹션 15 이후에 닫히는 단일
`ODS PDF`에 담긴다.

In [ ]:
제목 "ICIJ Paradise Papers — Headline Statistics";

처리 평균 데이터=생존 n mean std median p25 p75;
    변수 duration officer_count;
    제목 "Entity lifespan and officer-count summary (full n=12,130)";
실행;

처리 빈도 데이터=생존;
    tables top5;
    제목 "Jurisdiction distribution of surviving entities";
실행;


## 11. 임원 중심 위험 관점

섹션 2-10은 엔터티에 초점을 맞췄다. 그 엔터티 뒤에 있는 사람들 —
임원들 — 도 같은 주목을 받을 만하다. 컴플라이언스 실무는 *동시에*
(a) 많은 엔터티에 연결되어 있고, (b) 여러 관할구역에 걸쳐
활동하며, (c) 임원-엔터티 프로젝션에서 높은 PageRank로 존재하고,
(d) 긴 기간에 걸쳐 활동하는 임원을 그 자체로 구조적 위험으로
취급한다.

실제 그래프에서 임원별 특성 테이블을 구성한다:

| 특성 | 정의 |
|---|---|
| `degree` | 이 임원이 OFFICER_OF를 보유한 엔터티의 수 |
| `n_juris` | 그 엔터티들의 서로 다른 관할구역 수 |
| `pagerank` | 섹션 4에서 나온 임원 노드의 PageRank |
| `tenure_years` | 임원의 엔터티 전반에 걸친 `max(incorp_year)` − `min(incorp_year)` |

그런 다음 각 특성을 [0, 1]로 min-max 정규화하고 가중 합을 취한다
— 차수 30%, 로그 PageRank 30%, 관할구역 폭 20%, 재임 20% — 이를
단일 종합 **영향력 점수**로 삼는다. 이 점수 기준 상위 10명은 ICIJ
연구팀이 가장 많이 연결된 Appleby 행위자로 공개적으로 지목한
개인들을 드러낸다.

In [ ]:
처리 gql conn=icij out=임원_원본;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* 섹션 4 PROC NETWORK 출력에서 나온 PageRank 대응 중심성을
   임원 이름 기준 LEFT JOIN으로 붙인다. PROC SQL이 정렬+병합+
   coalesce를 한 번에 처리한다 — DATA MERGE BY 체인도, PROC SORT도
   필요 없다. */
처리 sql;
    create table 임원_특성 as
    선택 o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   임원_원본          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* 각 특성을 min-max 정규화하고 종합 영향력 점수를 구성하여
   영향력 상위 50개를 유지한다. 다단계 DATA 스텝 파이프라인 대신
   PROC RANK + PROC SQL을 사용한다. */
처리 평균 데이터=임원_특성 noprint;
    변수 degree pagerank n_juris tenure_years;
    출력 out=임원_최소최대
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
실행;

데이터 임원_점수;
    만약 _n_ = 1 이면 설정 임원_최소최대;
    설정 임원_특성;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    유지 node_id officer_name degree pagerank
         n_juris tenure_years influence;
실행;

처리 sql outobs=50;
    제목 "Section 11 — top-50 Paradise-Papers officers by composite influence";
    선택 officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   임원_점수
    order 기준 influence desc;
quit;

## 12. 시간적 설립 패턴

`incorporation_date`를 서버 측에서 네 자리 연도로 파싱하면, 지배적인
다섯 개 관할구역에 걸쳐 역외 설립 활동이 어떻게 진화했는지 볼 수
있다. 1970년부터 2014년까지 연간 신규 엔터티 수를 소형 다중 패널
`PROC SGPANEL` 레이아웃으로 그리면, 정책 분석가들이 찾는 종류의
법제 주도 급증이 드러난다.

실제 데이터에서:

- **케이맨 제도** 활동은 1990년대 후반부터 꾸준히 상승하여 2001년에
  연간 신규 엔터티 400개를 돌파하고, 2014년까지 연간 약 450-510개
  신규 엔터티로 안정세를 유지한다.
- **버뮤다**는 2007-2013년경 연간 210-290개로 정점을 찍으며, 전
  세계 헤지펀드 및 사모펀드 자금 조달 주기와 밀접하게 함께 움직인다.
- **영국령 버진아일랜드**는 2005년 연간 약 60개 신규 엔터티에서
  2014년 200개로 급격히 도약한다 — 파라다이스 페이퍼스가 다루는
  기간 동안 3.3배 증가.
- **맨섬**과 **저지**는 완만하게 유지되지만(연간 50-140개), 저지는
  2013년에 142개로 급격히 뛰어오른다 — 전체 기간에서 가장 높은
  저지 수치.

In [ ]:
처리 gql conn=icij out=연도_관할구역;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 상위 5개가 아닌 관할구역을 OTHER로 합친다. */
데이터 연도_패널;
    설정 연도_관할구역;
    길이 top5 $5;
    top5 = 'OTHER';
    만약 jurisdiction = 'BM' 이면 top5 = 'BM';
    아니면 만약 jurisdiction = 'KY' 이면 top5 = 'KY';
    아니면 만약 jurisdiction = 'VG' 이면 top5 = 'VG';
    아니면 만약 jurisdiction = 'IM' 이면 top5 = 'IM';
    아니면 만약 jurisdiction = 'JE' 이면 top5 = 'JE';
실행;

처리 평균 데이터=연도_패널 noprint nway;
    분류 year top5;
    변수 n;
    출력 out=연도_합계 (제거=_type_ _freq_)
        sum=entities;
실행;

처리 sgpanel 데이터=연도_합계;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis 라벨="Incorporation year";
    rowaxis 라벨="New entities per year";
    제목 "Section 12 — new entity formations per year, top-5 jurisdictions";
실행;

/* 상위 5개 + OTHER 전반의 정점 연도 급증 상위 3건. */
처리 정렬 데이터=연도_합계 out=연도_정점;
    기준 descending entities;
실행;

데이터 연도_정점;
    설정 연도_정점 (obs=10);
실행;

처리 인쇄 데이터=연도_정점;
    제목 "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
실행;

## 13. 유출 간 비교

파라다이스 페이퍼스 그래프는 ICIJ가 취합한 다섯 개의 독립적인 출처
스트림을 반영하여, `sourceID`에 따라 다섯 개의 하위 코퍼스로 내부
분할되어 있다:

- **Paradise Papers - Appleby** — Appleby 법무법인 유출(데이터의
  압도적 다수)
- **Paradise Papers - Malta corporate registry** — 몰타 공식
  법인 등기부의 유출본
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

각 `sourceID`를 하나의 "유출"로 취급하면, 각 코퍼스가 역외 세계의
자기 영역에 집중되어 있음을 확인할 수 있다. Appleby 유출은 압도적으로
버뮤다와 케이맨(명명된 엔터티의 합산 73%)이고, 몰타 유출은 사실상
모두 몰타 엔터티이며, 레바논 유출은 본질적으로 모두 레바논
엔터티인 식이다. 아래의 `PROC FREQ` 교차표는 이 집중을 가시화한다.

In [ ]:
처리 gql conn=icij out=유출_원본;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

처리 빈도 데이터=유출_원본 order=빈도;
    tables sourceid / out=유출_합계;
    가중 n;
    제목 "Section 13 — entity counts per sourceID corpus";
실행;

처리 인쇄 데이터=유출_합계;
    제목 "Section 13 — leak-level totals";
실행;

/* LIST 형식은 (유출, 관할구역) 셀마다 한 행을 출력한다 — 기본
   넓은 교차표 대신 터미널 너비에 맞춘다. */
처리 정렬 데이터=유출_원본 out=유출_정렬;
    기준 sourceid descending n;
실행;

처리 인쇄 데이터=유출_정렬 (obs=30);
    제목 "Section 13 — top 30 (leak, jurisdiction) cells";
실행;


## 14. 광범위한 제재 보강 — OpenSanctions

섹션 6의 OFAC-SDN 전용 스크리닝은 정확 매칭 적중이 0건이었다.
그것은 정직한 발견이었다 — 우리가 커밋한 500행 OFAC 표본은
압도적으로 2022년 RUSSIA-EO14024 프로그램에서 나왔고, 파라다이스
페이퍼스는 최신 기록이 2014년인 데이터로 편찬되었다.

그물을 넓히기 위해 이제 [OpenSanctions](https://www.opensanctions.org/)
*default* 통합 제재 데이터셋의 실제 발췌 — 다음 출처의 통합 공개
제재 목록의 2026-04-17 스냅샷 — 를 사용한다:

- 미국 OFAC 특별지정 대상자
- 영국 재무부 자산동결 대상
- EU 통합 금융 제재
- UN 안전보장이사회 제재
- 캐나다, 벨기에, 호주, 스위스, 노르웨이, 일본, 뉴질랜드,
  싱가포르 통합 목록

`data/opensanctions_default.csv`에 커밋된 하위 집합에는 주요
데이터셋이 선별된 핵심 제재 목록 중 하나인 18,654개의 Person 및
Company 레코드가 담겨 있다(BIC 및 FIRDS 식별자 카탈로그와 같은
참조 데이터 전용 출처는 제외하여 모든 적중이 진정한 제재 출처를
갖도록 함). 파일을 2 MB 미만으로 유지하기 위해 별칭은 삭제했다.

OpenSanctions 목록은 이전 OFAC 표본보다 한 자릿수 더 크기 때문에,
Neo4j에서 모든 Officer와 모든 Entity 이름을 *한 번* 가져와
`PROC SQL`로 제재 CSV와 로컬에서 조인한다. 정확한 UPCASE 매칭은
견고하며, 대규모 토큰 IN 목록을 괴롭히는 Cypher 문자열 길이 제한을
피한다.

In [ ]:
/* 함께 커밋된 OpenSanctions CSV를 읽는다. 헤더 주석 9줄과 열
   헤더 1줄 = firstobs=11. */
데이터 오픈제재;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    길이 os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    입력 os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    만약 길이(os_name_upper) >= 6;
실행;

처리 정렬 데이터=오픈제재 nodupkey out=제재_중복제거;
    기준 os_name_upper;
실행;

처리 평균 데이터=제재_중복제거 n;
    제목 "OpenSanctions sanctions-list records loaded";
실행;

/* 그래프에서 모든 임원 + 엔터티 이름을 가져온다. */
처리 gql conn=icij out=전체_임원;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

처리 gql conn=icij out=전체_엔터티;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* 정확한 UPCASE 매칭 — 임원과 제재 대상. */
처리 sql;
    create table s14_임원_적중 as
    선택 o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from 전체_임원  as o
    inner join 제재_중복제거 as s
    on o.officer_name_upper = s.os_name_upper;
quit;

처리 sql;
    create table s14_엔터티_적중 as
    선택 e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from 전체_엔터티 as e
    inner join 제재_중복제거 as s
    on e.entity_name_upper = s.os_name_upper;
quit;

처리 sql;
    선택 count(*) as n_officer_hits
    from s14_임원_적중;
    선택 count(*) as n_entity_hits
    from s14_엔터티_적중;
quit;

처리 인쇄 데이터=s14_임원_적중;
    제목 "Section 14 — officers on a consolidated sanctions list";
실행;

처리 인쇄 데이터=s14_엔터티_적중;
    제목 "Section 14 — entities on a consolidated sanctions list";
실행;

## 15. 종합 엔터티 위험 순위

마지막으로, 이전 섹션들에서 계산한 구조적, 관할구역적, 관계적,
제재 신호를 단일 엔터티별 종합 **위험 점수**로 결합한다:

| 구성 요소 | 가중치 | 출처 |
|---|---:|---|
| 임원 수(50으로 상한) | 0.25 | 섹션 5 특성 테이블 |
| log(1 + 대표 임원 PageRank) | 0.25 | 섹션 4 PageRank + 섹션 11 |
| 관할구역 위험 가중치 | 0.25 | Tax Justice Network CTHI 2021(커밋됨) |
| 제재 임원 플래그 | 0.25 | 섹션 14 정확 매칭 결과 |

관할구역 위험은 커밋된 파일 `data/tax_haven_ranks.csv`에서 나오며,
Tax Justice Network의 2021년 법인 조세피난처 지수 발표 순위 목록으로
구성된다. 순위 1-10은 2021 CTHI 보도자료에서 직접 가져왔고,
중위권 순위는 파라다이스 페이퍼스에 나타나는 나머지 관할구역에 대한
발표된 TJN 방법론 값이다. CTHI 순위가 없는 관할구역(예: `XX`
자리 표시자)은 가중치 ≈ 0을 받는다.

아래 보고서는 규제 당국을 위해 `PROC REPORT`로 스타일링되었다.
목록 상단의 엔터티는 동시에 (a) 첫 페이지 조세피난처 관할구역에
소재하고, (b) 임원이 많으며, (c) 최상위 PageRank 임원을 보유하고,
그리고 (d) 통합 제재 목록에 플래그된 임원이 최소 한 명 있는
엔터티이다.

In [ ]:
/* 함께 커밋된 TJN 법인 조세피난처 지수 2021 순위를 읽는다.
   주석 8줄 + // 2줄 + 헤더 1줄 = firstobs=16. */
데이터 조세피난처;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    길이 iso2 $5 jurisdiction_name $50;
    입력 iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
실행;

처리 인쇄 데이터=조세피난처 (obs=10);
    제목 "Section 15 — jurisdiction risk weights (CTHI 2021)";
실행;

/* 대표 임원 이름과 설립 연도를 포함한 엔터티별 특성. */
처리 gql conn=icij out=엔터티_전체;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* 함께 결합해야 하는 모든 것(pagerank, 위험 가중치, 제재 임원
   플래그)을 단일 PROC SQL 3방향 LEFT JOIN으로 처리한다 — DATA
   MERGE BY 체인도, 불필요한 PROC SORT도 없고, COALESCE로 대체
   값을 인라인으로 제공한다. */
처리 gql conn=icij out=표시_엔터티_id;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

처리 sql;
    create table 엔터티_표시 as
    선택 e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case 경우 f.node_id is 아닌 null 이면 1 아니면 0 종료
               as sanctioned_officer
    from   엔터티_전체        as e
    left join 임원_점수   as p
      on   e.top_officer_name = p.officer_name
    left join 조세피난처       as t
      on   e.jurisdiction     = t.iso2
    left join 표시_엔터티_id as f
      on   e.node_id          = f.node_id;
quit;

/* 종합 위험: 연속형 특성을 min-max 정규화한 뒤 가중 합산한다.
   PROC MEANS + 단일 DATA 스텝으로 정규화하는 것은 관용적인 SAS
   방식이다. */
처리 평균 데이터=엔터티_표시 noprint;
    변수 top_officer_pr;
    출력 out=페이지랭크_최대셋 max=pr_max;
실행;

데이터 엔터티_점수;
    만약 _n_ = 1 이면 설정 페이지랭크_최대셋;
    설정 엔터티_표시;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    유지 node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
실행;

/* 전체 24,957개 엔터티 모집단에 걸친 위험 분포. */
처리 평균 데이터=엔터티_점수 n min mean max;
    변수 risk officer_count risk_weight;
    제목 "Section 15 — risk distribution across all entities";
실행;

/* 종합 위험 상위 25개 엔터티. PROC SQL OUTOBS=가 PROC SORT +
   DATA 스텝 obs= 조합을 대체한다. */
처리 sql outobs=25;
    제목 "Section 15 — top-25 composite-risk entities (names)";
    선택 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   엔터티_점수
    order 기준 risk desc;
quit;

/* 제재 임원이 연결된 엔터티를 별도로 표시한다. */
처리 sql;
    제목 "Section 15 — entities with at least one sanctioned officer";
    선택 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   엔터티_점수
    조건  sanctioned_officer = 1
    order 기준 risk desc;
quit;

## 16. 도관 대 종착지 관할구역 분류

**참고문헌:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo 등은 전 세계 법인 소유 그래프를 "종착지-OFC"
(법인 가치가 종결되는 곳: BM, KY, BVI, JE, IM)와 "도관-OFC"
(가치가 흐르는 통로: NL, UK, CH, SG, IE)로 분할한다. 파라다이스
페이퍼스 그래프는 모집단이 다르지만(대부분 Appleby 소재 엔터티),
동일한 구조적 구분은 임원이 모이고 주소가 종결되는 관할구역과
주로 자본을 통과시키는 관할구역을 분리해야 한다.

각 관할구역에 대해 라이브 그래프에서 직접 다섯 개의 구조적 특성을
계산한다:

| 특성 | 신호 |
|---|---|
| `log(n_entity)` | 관할구역 역외 존재의 절대 규모 |
| `avg_officers` | 엔터티당 임원 밀도(종착지 관할구역은 엔터티당 임원이 더 많음) |
| `avg_xborder_off` | 자국이 엔터티 관할구역과 다른 임원의 평균 수(도관 지표) |
| `intermediary_share` | CONNECTED_TO 중개기관 링크를 가진 엔터티의 비율 |
| `address_share` | REGISTERED_ADDRESS 링크를 가진 엔터티의 비율(종착지 지표) |

z-점수로 표준화한 뒤 Ward 최소 분산 계층적 군집화를 적용한다.
`PROC TREE`가 덴드로그램을 렌더링한다. 파라다이스 페이퍼스의
Intermediary 노드는 `INTERMEDIARY_OF`가 아니라 `CONNECTED_TO`를
통해 Entity 노드에 연결되어 있음에 유의하라 —
`INTERMEDIARY_OF`는 스키마에는 존재하지만 이 유출에는 데이터가
없다 — 따라서 여기서는 `CONNECTED_TO`를 사용한다.

In [ ]:
/* 라이브 그래프에서 관할구역별 구조적 특성을 가져온다. */
처리 gql conn=icij out=s16_원본;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* 표준화된 z-점수가 의미를 갖도록 엔터티가 최소 10개 이상인
   관할구역만 유지한다. 파라다이스 페이퍼스 유출에는 총 44개
   관할구역이 있고, 그중 23개가 이 임계값을 충족한다. */
데이터 s16_필터;
    설정 s16_원본;
    만약 n_entity >= 10;
    log_n_entity = log(n_entity);
실행;

처리 평균 데이터=s16_필터 noprint;
    변수 log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    출력 out=s16_통계
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
실행;

데이터 s16_표준화;
    만약 _n_ = 1 이면 설정 s16_통계;
    설정 s16_필터;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    유지 jurisdiction z1 z2 z3 z4 z5;
실행;

처리 인쇄 데이터=s16_표준화;
    제목 "Section 16 — standardised jurisdiction features";
실행;

/* Ward 최소 분산 계층적 군집화. */
처리 cluster 데이터=s16_표준화 method=ward outtree=s16_트리;
    변수 z1 z2 z3 z4 z5;
    id jurisdiction;
    제목 "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
실행;

/* 덴드로그램을 렌더링한다. */
처리 tree 데이터=s16_트리 horizontal;
    id jurisdiction;
    제목 "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
실행;

## 17. 임원 네트워크 역할의 주성분

**참고문헌:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
See also Newman M E J, *Networks: An Introduction* (Oxford, 2010),
chapter 7.

파라다이스 페이퍼스 그래프의 임원들은 구조적으로 서로 다른 역할을
한다. 어떤 임원은 관련 엔터티의 큰 클러스터 중심에 자리하고, 다른
임원은 서로 분리된 클러스터들을 연결하며(Burt/Borgatti 의미의
브로커), 소수는 특정 관할구역 내부자 네트워크의 밀집된 핵심을
형성한다. 네 가지 그래프 지표가 이 뚜렷한 역할들을 포착한다:

| 지표 | 포착하는 것 |
|---|---|
| `degree` | `OFFICER_OF` 나가는 엣지 수 — 제휴의 폭 |
| `betweenness` | 임원을 통과하는 최단 경로의 비율 — **브로커리지** |
| `kcore` | 임원이 속한 k-연결 하위 그래프의 최대 k — **핵심 배태성** |
| `pagerank` | 동일 프로젝션에서의 고유벡터형 점수 — **영향력 있는 파트너를 통한 영향력** |

Neo4j Graph Data Science 라이브러리를 통해 전체
`(Officer)—[OFFICER_OF]—(Entity)` 무방향 프로젝션에서 네 지표를
모두 계산하고, 차수 상위 5,000명 임원으로 제한한 뒤, 로그 변환된
지표에 대해 `PROC PRINCOMP`을 실행한다.

PCA는 네 개의 상관된 지표를 직교 축으로 압축하며, 그 상대적
적재값은 어떤 역할들이 경험적으로 함께 묶이고 어떤 역할이 구조적으로
구별되는지 알려준다.

*국소 군집 계수에 대한 참고:* Garcia-Bernardo 등은 국소 군집
계수를 다섯 번째 지표로 포함한다. 파라다이스 페이퍼스
`(Officer)—[:OFFICER_OF]—(Entity)` 프로젝션은 엄밀히 이분적이므로
삼각형이 존재할 수 없다 — 모든 국소 군집 계수가 0이다. 우리는 이를
지표 집합에서 제외한다.

In [ ]:
/* PROC NETWORK은 읽기 전용 MATCH로 상위 5000명 임원 하위
   그래프를 가져와 차수, 고유벡터 중심성, k-core를 인프로세스로
   계산한다. 이전의 gds.graph.project + 네 번의
   gds.<algorithm>.stream 호출을 대체한다 — 그 패턴은 플랫폼의
   읽기 전용 step-neo4j 서비스가 거부하는 GDS 쓰기 모드 프로젝션
   단계를 요구한다.

   매개 중심성은 의도적으로 생략한다: NetworkX는 정확한 매개
   중심성을 O(V·E)로 계산하는데, 이 하위 그래프(임원 5,000명 ×
   약 78,000 엣지)에서는 이것이 실행 시간을 지배한다. PCA는 여전히
   세 개의 직교 축 — 차수(국소적 두드러짐), 고유벡터 중심성(전역
   영향력), k-core(구조적 응집) — 을 가지며, 핵심 해석을 위해
   임원 역할을 구분하기에 충분하다. */
처리 network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_지표_전체;
    linksvar from=a_node_id 까지=b_node_id;
    centrality degree eigen=unweight;
    core;
실행;

/* 필터링을 위해 임원 노드 id/이름을 가져온다. */
처리 gql conn=icij out=전체_임원_이름;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* 임원 행으로 필터링한다. 고유벡터 중심성이 PageRank를 대신한다
   — 섹션 4 설명 참조. */
처리 sql;
    create table s17_지표 as
    선택 n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_지표_전체 as n
    inner join 전체_임원_이름 as o on n.node = o.node_id
    order 기준 n.centr_degree desc;
quit;

데이터 s17_지표;
    설정 s17_지표;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    유지 node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
실행;

처리 인쇄 데이터=s17_지표 (obs=10);
    제목 "Section 17 — top-10 officers by degree (raw + log metrics)";
실행;

처리 평균 데이터=s17_지표 n mean std min max;
    변수 log_degree log_pr k_val;
    제목 "Section 17 — log-transformed metric summary";
실행;

처리 princomp 데이터=s17_지표 out=s17_점수;
    변수 log_degree log_pr k_val;
    제목 "Section 17 — PCA of officer network roles";
실행;

처리 sgplot 데이터=s17_점수;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis 라벨="PC1 (global influence axis)";
    yaxis 라벨="PC2 (brokerage vs core embeddedness)";
    제목 "Section 17 — officers in 2D principal-component role space";
실행;

## 18. 설립률에 대한 ARIMA 개입 분석

**참고문헌:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Applied to offshore-leaks by Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

파라다이스 페이퍼스 그래프의 연간 신규 엔터티 수는 1970년(36개
엔터티)부터 2007년(1,595개 엔터티, 정점)까지 거의 단조적으로
증가하는 시계열이며, 이후 2008-2009년 하락과 2014년(유출 범위의
끝)까지의 완만한 안정세가 뒤따른다.

우리는 고전적인 Box-Tiao 개입 분석을 적용하여 두 개의 실제 사건이
설립 시계열에 측정 가능한 흔적을 남겼는지 검정한다:

- **2009년 계단** — G20 런던 정상회의의 조세피난처 단속(2009년
  4월)으로, 금융위기와 시기가 겹쳤다.
- **2014년 계단** — 미국 FATCA(해외금융계좌신고법)가 2014년
  7월 1일 발효되었다.

반응은 `log(n)`이므로 개입 계수 -0.30은 연간 설립률의 약 26%
감소에 해당한다. 강하게 상관된 시계열에 대한 AR(1) 자기회귀
모형인 `ARIMA(1,0,0)`을 두 계단 더미를 외생 `INPUT=` 변수로 하여
적합시킨다.

귀무가설은 AR(1) 추세를 고려하면 어느 계단도 측정 가능한 이동을
일으키지 않는다는 것이다. 발표된 방법론들은 비기각을 어떻게
해석할지에 대해 견해가 갈린다: 개입에 효과가 없었을 수도 있고,
AR(1) 자기상관이 개입 신호를 흡수하고 있을 수도 있다.

In [ ]:
처리 gql conn=icij out=연도_개수;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* 개입 시계열 데이터셋을 구성한다. 두 개의 계단 더미:
   step_2009 = I{year >= 2009}는 G20 런던/FATCA 사전 발표 체제
   변화를 포착하고, step_2014 = I{year >= 2014}는 FATCA 발효일
   충격을 포착한다. 반응은 log(n)이므로 계수 추정치는 비례 효과로
   읽힌다. */
데이터 s18_시계열;
    설정 연도_개수;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
실행;

처리 인쇄 데이터=s18_시계열;
    제목 "Section 18 — annual new-entity formations + intervention dummies";
실행;

처리 sgplot 데이터=s18_시계열;
    series x=year y=n / markers;
    refline 2009 / axis=x 라벨="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x 라벨="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis 라벨="Incorporation year";
    yaxis 라벨="New entities per year";
    제목 "Section 18 — Paradise-Papers annual formations, 1970-2014";
실행;

/* 모형을 식별한 뒤 두 계단 입력으로 ARIMA(1,0,0)을 추정한다.
   PROC ARIMA의 CROSSCORR=는 IDENTIFY 단계에서 외생 변수를
   등록하여 ESTIMATE INPUT=에서 사용할 수 있게 한다. */
처리 arima 데이터=s18_시계열;
    identify 변수=log_n crosscorr=(step_2009 step_2014) nlag=8;
    추정 p=1 입력=(step_2009 step_2014);
    제목 "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
실행;
quit;

## 19. 영과잉 제재 노출 카운트 모형

**참고문헌:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2nd edition, Cambridge University Press (2013), chapter 4.
Zero-inflated models: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

섹션 14는 통합 제재 목록에 오른 임원이 최소 한 명 있는 파라다이스
페이퍼스 엔터티를 **다섯 개**만 발견했다 — 극히 드문 사건이다.
엔터티별 `sanctioned_count`에 대한 표준 포아송 또는 음이항 회귀는
엔터티의 **99.98%**가 0이기 때문에 잘못 적합될 것이다. 영과잉
음이항(ZINB) 모형은 적합을 두 부분으로 나누어 이를 처리한다:

1. 엔터티가 "구조적 영" 부류(제재 노출 가능성이 없음)에 속하는지에
   대한 로지스틱 모형.
2. 나머지 중 카운트에 대한 음이항 모형.

21,538개 엔터티에 걸쳐 양성 사건이 5건뿐이라 ZINB 우도는 퇴화한다
— 두 부분 모두 붕괴한다. 그 실패는 절차가 아니라 **데이터의 정직한
속성**이다. 회귀 도구가 엔드투엔드로 작동함을 보이기 위해 ZINB
적합을 그래도 실행한 뒤, `has_sanctioned`(`sanctioned_count > 0`
지표)에 대한 일반 이진 로지스틱 회귀로 대체한다. 로지스틱은 깨끗한
효과를 식별한다: **임원 수의 로그 단위가 하나 늘 때마다 제재 임원을
최소 한 명 보유할 오즈가 약 3.1배 증가한다**(p = 0.002).

공변량:

- `top5` — 6수준 클래스 변수(BM, KY, VG, IM, JE, OTHER), 기준
  범주 OTHER.
- `log_n_off` — `log(officer_count)`, 지배적인 규모 예측 변수.

In [ ]:
/* 라이브 그래프에서 엔터티별 제재 임원 수를 가져온다. */
처리 gql conn=icij out=s19_원본;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

데이터 s19;
    설정 s19_원본;
    만약 officer_count >= 1;
    길이 top5 $5;
    top5 = 'OTHER';
    만약 jurisdiction = 'BM' 이면 top5 = 'BM';
    아니면 만약 jurisdiction = 'KY' 이면 top5 = 'KY';
    아니면 만약 jurisdiction = 'VG' 이면 top5 = 'VG';
    아니면 만약 jurisdiction = 'IM' 이면 top5 = 'IM';
    아니면 만약 jurisdiction = 'JE' 이면 top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
실행;

처리 빈도 데이터=s19;
    tables top5 sanctioned_count has_sanctioned;
    제목 "Section 19 — response-variable distribution (very zero-heavy)";
실행;

/* ZINB 먼저 — 5건짜리 시계열에서는 퇴화될 것으로 예상됨. */
처리 genmod 데이터=s19;
    분류 top5;
    모형 sanctioned_count = top5 log_n_off / dist=zinb link=log;
    제목 "Section 19 — ZINB count model (degenerate on 5 events)";
실행;

/* 대체 수단: has_sanctioned에 대한 이진 로지스틱. 해석 가능. */
처리 logistic 데이터=s19 descending plots=none;
    분류 top5;
    모형 has_sanctioned = top5 log_n_off;
    제목 "Section 19 — logistic regression (has-sanctioned fallback)";
실행;

## 20. 혼합효과 포아송 설립률 모형

**참고문헌:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Panel-count classical: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

섹션 18은 집계된 설립 시계열에 단변량 ARIMA를 적합시켰다. 여기서는
**패널** 차원을 사용한다: 관할구역-연도 셀당 한 행으로, 고정 선형
연도 추세에 FATCA 계단 더미를 더하고 **관할구역별 임의 절편**을
갖는 포아송 일반화 선형 혼합 모형(GLMM)을 적합시킨다. 이는 공통
추세 효과를 개별 관할구역의 수준에서 분리한다.

패널 제한: 1990-2014년 동안 **연평균 카운트**가 5 이상인 10개
관할구역(총 203개 셀). 영 카운트 연도가 많은 소규모 관할구역은
포아송 적합을 불안정하게 만든다.

`DIST=POISSON LINK=LOG`와 `RANDOM INTERCEPT / SUBJECT=jurisdiction`을
가진 `PROC GLIMMIX`는 모집단 수준 고정 효과(연도 추세, FATCA
이동)와 관할구역 간 분산 성분을 모두 산출한다. 임의 절편 분산은
*공통 시간 추세를 고려한 뒤 관할구역 간 설립률이 얼마나 다른지*를
알려준다 — 역외 금융 중심지 모집단에 대한 구조적 이질성 점수이다.

In [ ]:
/* 패널 데이터셋: 1990-2014의 관할구역 x 연도 셀. */
처리 gql conn=icij out=s20_원본;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 연평균 카운트 >= 5인 관할구역만 유지한다. */
처리 sql;
    create table s20_유지_관할구역 as
    선택 jurisdiction, avg(n) as avg_n
    from s20_원본
    group 기준 jurisdiction
    having avg(n) >= 5;
quit;

처리 sql;
    create table s20 as
    선택 r.*,
           r.year - 2002 as year_c,
           case 경우 r.year >= 2014 이면 1 아니면 0 종료 as fatca
    from s20_원본 r
    inner join s20_유지_관할구역 k
    on r.jurisdiction = k.jurisdiction;
quit;

처리 빈도 데이터=s20;
    tables jurisdiction;
    제목 "Section 20 — jurisdictions retained in the panel";
실행;

/* 혼합효과 포아송 GLMM: 고정 연도 추세 + FATCA 계단, 관할구역별
   임의 절편. */
처리 glimmix 데이터=s20;
    분류 jurisdiction;
    모형 n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    제목 "Section 20 — mixed Poisson formation-rate model";
실행;

/* 순위가 매겨진 임의 절편 효과는 공통 추세를 체계적으로
   상회하거나 하회하는 관할구역을 드러낼 것이다. PROC GLIMMIX
   SOLUTION 문이 위의 기본 출력에 이를 인쇄하므로, 순위 매김은
   독자에게 맡긴다. */

In [ ]:
/* 보고서 PDF를 닫고 Neo4j 라이브러리를 해제한다. */
ods pdf close;

libname icij clear;

## 재현성 및 출처

- **그래프 데이터 출처:** ICIJ *Offshore Leaks* 연구 데이터베이스,
  *Paradise Papers* 데이터셋, 2017년 11월 최초 공개.
  <https://offshoreleaks.icij.org/pages/database>에서 이용 가능.
  상류 공개 덤프(`github.com/neo4j-graph-examples/icij-paradise-papers`)를
  통해 플랫폼의 공유 `step-neo4j` 서비스(Neo4j 5.26 Community
  Edition, 서버 수준 읽기 전용)에 적재됨.
- **OFAC SDN 데이터:** 미국 재무부 OFAC 특별지정 대상자 공개 CSV
  익스포트로, 2026년 4월 재무부 공개 프리뷰 API에서 가져옴.
  `data/ofac_sdn.csv` 파일에는 항목 수 기준 가장 큰 다섯 개 OFAC
  프로그램에 걸쳐 선별한 500행이 담겨 있다. 섹션 6 2단계 스크린에
  사용됨.
- **OpenSanctions 데이터:** OpenSanctions *default* 통합 데이터셋
  스냅샷(2026-04-17), `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`에서
  다운로드. 커밋된 파일 `data/opensanctions_default.csv`에는 주요
  데이터셋이 OFAC SDN, 영국 재무부, EU 금융 제재, UN 안전보장이사회,
  캐나다, 벨기에, 호주, 스위스 또는 기타 국가 통합 제재 목록 중
  하나인 18,654개의 Person 및 Company 스키마 행이 담겨 있다. 파일을
  2 MB 미만으로 유지하기 위해 별칭은 삭제했다. 라이선스: ODbL 1.0
  (OpenSanctions). 섹션 14 보강에 사용됨.
- **조세피난처 순위:** Tax Justice Network *법인 조세피난처 지수
  2021* 발표 순위로, `https://cthi.taxjustice.net` 지수 랜딩
  페이지와 2021년 3월 보도자료
  (`https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`)에서
  편찬됨. 커밋된 파일 `data/tax_haven_ranks.csv`는 파라다이스
  페이퍼스에 나타나는 관할구역과 그 CTHI 순위, 그리고 파생된
  `[0, 1]` 위험 가중치를 나열한다. 라이선스: CC BY-SA 4.0 (Tax
  Justice Network). 섹션 15 종합 순위에 사용됨.
- **그래프 알고리즘:** Louvain 커뮤니티 탐지와 고유벡터 중심성
  (PageRank에 가장 가까운 자체 대응물)은 읽기 전용 Cypher로 가져온
  엣지 목록에 대해 `PROC NETWORK`이 인프로세스로 계산한다. 플랫폼의
  공유 `step-neo4j` 서비스는 서버 수준 읽기 전용이므로, 모든 그래프
  알고리즘은 Neo4j Graph Data Science 쓰기 작업이 아니라 workspace
  파드에서 실행된다.
- **통계 방법:** `PROC LIFETEST`는 Kaplan-Meier 추정량과 로그순위,
  Wilcoxon, Tarone-Ware 층화 검정을 사용한다. `PROC PHREG`는
  Python/statsmodels 래퍼를 사용해 Breslow 동점 처리로 Cox
  비례위험 모형을 적합한다. `PROC LOGISTIC`은 최대우도 이진 로지스틱
  회귀를 적합한다.
- **위험 구성 방법:** 섹션 11 종합 영향력 점수는 차수, 로그
  PageRank, 관할구역 폭, 재임 연수를 `[0, 1]`로 정규화하고 고정
  가중치(30/30/20/20)로 합산한다. 섹션 15 종합 엔터티 위험 점수는
  상한 처리된 임원 수, 로그 PageRank, CTHI 위험 가중치, 이진 제재
  임원 플래그를 정규화하고 각각 0.25의 동일 가중치로 합산한다.

전체 분석은 이 폴더의 스모크 테스트 스크립트 `./smoke.jenner`로
재현할 수 있다. 공유 `step-neo4j` 서비스를 대상으로(플랫폼이 모든
workspace 파드에서 대신 설정해 주는 `JENNER_NEO4J_HOST`와
`JENNER_NEO4J_PASS`가 설정된 상태로) 엔드투엔드 실행하는 데 약 2분이
걸리며, 기존 파이프라인과 함께 추가된 다섯 개의 새 섹션을 포함하여
모든 쿼리와 모든 PROC가 실제 163,435개 노드 그래프에서 기대되는
출력을 반환함을 검증한다.